# Customer Lifetime Value (CLTV) Analysis

## Objective

The objective of this notebook is to estimate the historical Customer Lifetime Value (CLTV) of customers using transactional purchasing behavior.

The analysis includes:

- Revenue feature engineering
- Customer purchase frequency
- Average order value
- Customer lifespan estimation
- Historical CLTV calculation
- Customer segmentation
- Executive visualizations

The outputs generated in this notebook will support executive reporting and Power BI dashboard development.

In [1]:
# ==========================================
# Import Libraries
# ==========================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
# ==========================================
# Project Paths
# ==========================================

PROJECT_ROOT = Path.cwd().parent

RAW_DATA = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"
VISUALS = PROJECT_ROOT / "visuals"

In [3]:
# ==========================================
# Load Clean Dataset
# ==========================================

df = pd.read_csv(
    PROCESSED_DATA / "online_retail_clean.csv",
    parse_dates=["InvoiceDate"]
)

print(df.shape)
df.head()

(779425, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,"13,085.00",United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,"13,085.00",United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,"13,085.00",United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,"13,085.00",United Kingdom


In [4]:
# ==========================================
# Data Validation
# ==========================================

print("Shape:", df.shape)
print("Missing Customer IDs:", df["Customer ID"].isna().sum())
print("Negative Quantity:", (df["Quantity"] < 0).sum())
print("Invalid Prices:", (df["Price"] <= 0).sum())
print("Cancelled Invoices:", df["Invoice"].astype(str).str.startswith("C").sum())

Shape: (779425, 8)
Missing Customer IDs: 0
Negative Quantity: 0
Invalid Prices: 0
Cancelled Invoices: 0


In [5]:
# ==========================================
# Create Revenue Feature
# ==========================================

df["Revenue"] = df["Quantity"] * df["Price"]

df[["Quantity", "Price", "Revenue"]].head()

,Quantity,Price,Revenue
0,12,6.95,83.40
1,12,6.75,81.00
2,12,6.75,81.00
3,48,2.10,100.80
4,24,1.25,30.00


In [6]:
# ==========================================
# Revenue Summary
# ==========================================

df["Revenue"].describe().round(2)

count   779,425.00
mean         22.29
std         227.43
min           0.00
25%           4.95
50%          12.48
75%          19.80
max     168,469.60
Name: Revenue, dtype: float64

In [7]:
# ==========================================
# Customer-Level Summary
# ==========================================

customer_summary = (
    df.groupby("Customer ID")
      .agg(
          TotalRevenue=("Revenue", "sum"),
          TotalOrders=("Invoice", "nunique"),
          TotalTransactions=("Invoice", "count"),
          TotalProducts=("Quantity", "sum"),
          FirstPurchase=("InvoiceDate", "min"),
          LastPurchase=("InvoiceDate", "max")
      )
      .reset_index()
)

customer_summary.head()

,Customer ID,TotalRevenue,TotalOrders,TotalTransactions,TotalProducts,FirstPurchase,LastPurchase
0,"12,346.00","77,556.46",12,34,74285,2009-12-14 08:34:00,2011-01-18 10:01:00
1,"12,347.00","4,921.53",8,222,2967,2010-10-31 14:20:00,2011-12-07 15:52:00
2,"12,348.00","2,019.40",5,51,2714,2010-09-27 14:59:00,2011-09-25 13:13:00
3,"12,349.00","4,428.69",4,175,1624,2010-04-29 13:20:00,2011-11-21 09:51:00
4,"12,350.00",334.40,1,17,197,2011-02-02 16:01:00,2011-02-02 16:01:00


In [8]:
print(customer_summary.shape)

customer_summary.describe().round(2)

(5878, 7)


,Customer ID,TotalRevenue,TotalOrders,TotalTransactions,TotalProducts,FirstPurchase,LastPurchase
count,"5,878.00","5,878.00","5,878.00","5,878.00","5,878.00",5878,5878
mean,"15,315.31","2,955.90",6.29,132.60,"1,788.70",2010-08-22 06:57:33.297039,2011-05-22 16:19:59.469207
min,"12,346.00",2.95,1.00,1.00,1.00,2009-12-01 07:45:00,2009-12-01 09:55:00
25%,"13,833.25",342.28,1.00,20.00,187.00,2010-02-09 14:01:15,2010-11-25 10:24:45
50%,"15,314.50",867.74,3.00,52.00,480.00,2010-06-27 13:31:30,2011-09-05 11:59:00
75%,"16,797.75","2,248.30",7.00,138.00,"1,350.00",2011-01-30 14:30:15,2011-11-14 11:31:15
max,"18,287.00","580,987.04",398.00,"12,435.00","367,193.00",2011-12-09 12:16:00,2011-12-09 12:50:00
std,"1,715.57","14,440.85",13.01,342.19,"8,876.30",NaN,NaN


## Customer-Level Summary

A customer-level analytical dataset was created by aggregating individual transaction records into one record per customer. The summary includes total revenue, order count, transaction volume, quantity of products purchased, and purchase history.

### Key Findings

- The dataset contains **5,878 unique customers**.
- Average customer revenue is **2,955.90**, while the median revenue is **867.74**, indicating a highly right-skewed revenue distribution.
- Customer purchasing behavior varies substantially, with orders ranging from **1** to **398**.
- A small number of customers contribute exceptionally high revenues, suggesting the presence of high-value or wholesale customers.
- The customer summary provides the analytical foundation for calculating Average Order Value (AOV), Purchase Frequency, Customer Lifespan, and Historical Customer Lifetime Value (CLTV).

In [9]:
# ==========================================
# Calculate Customer Lifespan
# ==========================================

customer_summary["CustomerLifespan"] = (
    customer_summary["LastPurchase"] -
    customer_summary["FirstPurchase"]
).dt.days

customer_summary["CustomerLifespan"].describe().round(2)

count   5,878.00
mean      273.02
std       258.81
min         0.00
25%         0.00
50%       220.50
75%       511.00
max       738.00
Name: CustomerLifespan, dtype: float64

## Customer Lifespan Analysis

Customer lifespan was calculated as the number of days between each customer's first and last recorded purchases.

### Key Findings

- The average customer lifespan is **273 days** (approximately nine months).
- The median lifespan is **220 days**, indicating that half of the customers remained active for more than seven months.
- Approximately **25% of customers made only one purchase**, resulting in a lifespan of zero days.
- The longest customer relationship lasted **738 days**, demonstrating strong long-term customer retention for a subset of customers.

These findings highlight significant differences in customer loyalty and purchasing behavior, emphasizing the importance of Customer Lifetime Value (CLTV) analysis for identifying high-value, long-term customers.

In [10]:
customer_summary[
    ["FirstPurchase", "LastPurchase", "CustomerLifespan"]
].head()

,FirstPurchase,LastPurchase,CustomerLifespan
0,2009-12-14 08:34:00,2011-01-18 10:01:00,400
1,2010-10-31 14:20:00,2011-12-07 15:52:00,402
2,2010-09-27 14:59:00,2011-09-25 13:13:00,362
3,2010-04-29 13:20:00,2011-11-21 09:51:00,570
4,2011-02-02 16:01:00,2011-02-02 16:01:00,0


## Customer Lifespan Calculation

Customer lifespan was calculated as the number of days between each customer's first and last recorded purchases.

### Observations

- Customers who made repeat purchases have positive lifespan values, reflecting the duration of their relationship with the business.
- Customers with a lifespan of **0 days** are one-time buyers whose first and last purchases occurred on the same day.
- The calculated lifespan will be used in the Historical Customer Lifetime Value (CLTV) model to estimate long-term customer value.

In [11]:
# ==========================================
# Average Order Value
# ==========================================

customer_summary["AverageOrderValue"] = (
    customer_summary["TotalRevenue"] /
    customer_summary["TotalOrders"]
)

customer_summary["AverageOrderValue"].describe().round(2)

count    5,878.00
mean       385.18
std      1,214.29
min          2.95
25%        176.68
50%        279.24
75%        414.90
max     84,236.25
Name: AverageOrderValue, dtype: float64

In [12]:
customer_summary[
    [
        "Customer ID",
        "TotalRevenue",
        "TotalOrders",
        "AverageOrderValue"
    ]
].head()

,Customer ID,TotalRevenue,TotalOrders,AverageOrderValue
0,"12,346.00","77,556.46",12,"6,463.04"
1,"12,347.00","4,921.53",8,615.19
2,"12,348.00","2,019.40",5,403.88
3,"12,349.00","4,428.69",4,"1,107.17"
4,"12,350.00",334.40,1,334.40


## Average Order Value (AOV)

Average Order Value (AOV) was calculated by dividing each customer's total revenue by the total number of orders placed.

### Business Insight

- Customers with higher AOV generate more revenue per transaction.
- AOV helps distinguish customers who make fewer, high-value purchases from those who purchase more frequently with smaller order values.
- Together with purchase frequency and customer lifespan, AOV forms a core component of Historical Customer Lifetime Value (CLTV).

In [13]:
total_customers = customer_summary.shape[0]

print(total_customers)

5878


In [14]:
# ==========================================
# Calculate Purchase Frequency
# ==========================================

total_customers = customer_summary.shape[0]
total_orders = customer_summary["TotalOrders"].sum()

purchase_frequency = total_orders / total_customers

print(f"Total Customers      : {total_customers:,}")
print(f"Total Orders         : {total_orders:,}")
print(f"Purchase Frequency   : {purchase_frequency:.2f}")

Total Customers      : 5,878
Total Orders         : 36,969
Purchase Frequency   : 6.29


In [15]:
customer_summary["PurchaseFrequency"] = purchase_frequency

customer_summary[
    [
        "Customer ID",
        "TotalOrders",
        "PurchaseFrequency"
    ]
].head()

,Customer ID,TotalOrders,PurchaseFrequency
0,"12,346.00",12,6.29
1,"12,347.00",8,6.29
2,"12,348.00",5,6.29
3,"12,349.00",4,6.29
4,"12,350.00",1,6.29


## Purchase Frequency

Purchase Frequency was calculated as the average number of orders placed per customer across the entire customer base.

### Formula

Purchase Frequency = Total Orders ÷ Total Customers

### Key Findings

- Total Customers: **5,878**
- Total Orders: **36,969**
- Average Purchase Frequency: **6.29 orders per customer**

This metric reflects the overall purchasing behaviour of the customer base and is used as a key component in estimating Historical Customer Lifetime Value (CLTV).

In [16]:
# ==========================================
# Customer Age
# ==========================================

snapshot_date = df["InvoiceDate"].max()

customer_summary["CustomerAge"] = (
    snapshot_date -
    customer_summary["FirstPurchase"]
).dt.days

customer_summary["CustomerAge"].describe().round(2)

count   5,878.00
mean      473.71
std       223.10
min         0.00
25%       312.00
50%       529.00
75%       667.00
max       738.00
Name: CustomerAge, dtype: float64

In [17]:
customer_summary[
    [
        "Customer ID",
        "FirstPurchase",
        "CustomerAge"
    ]
].head()

,Customer ID,FirstPurchase,CustomerAge
0,"12,346.00",2009-12-14 08:34:00,725
1,"12,347.00",2010-10-31 14:20:00,403
2,"12,348.00",2010-09-27 14:59:00,437
3,"12,349.00",2010-04-29 13:20:00,588
4,"12,350.00",2011-02-02 16:01:00,309


## Customer Age Analysis

Customer Age was calculated as the number of days between each customer's first recorded purchase and the latest transaction date in the dataset.

### Key Findings

- The average customer age is **473.71 days**, indicating that customers remained in the dataset for approximately 1.3 years on average.
- The median customer age is **529 days**, suggesting that at least half of the customers have maintained a relationship with the business for more than one year.
- Some customers joined near the end of the observation period, resulting in a customer age of zero days.
- Customer Age complements Customer Lifespan by measuring customer tenure within the observation window and is useful for interpreting long-term customer relationships.

In [18]:
# ==========================================
# Calculate Historical CLTV
# ==========================================

customer_summary["HistoricalCLTV"] = (
    customer_summary["AverageOrderValue"]
    * customer_summary["PurchaseFrequency"]
    * customer_summary["CustomerLifespan"]
)

customer_summary["HistoricalCLTV"] = customer_summary["HistoricalCLTV"].round(2)

customer_summary["HistoricalCLTV"].describe().round(2)

count         5,878.00
mean        691,695.66
std       1,787,854.51
min               0.00
25%               0.00
50%         323,894.38
75%         950,187.68
max     108,078,003.56
Name: HistoricalCLTV, dtype: float64

In [19]:
customer_summary[
    [
        "Customer ID",
        "AverageOrderValue",
        "PurchaseFrequency",
        "CustomerLifespan",
        "HistoricalCLTV"
    ]
].head()

,Customer ID,AverageOrderValue,PurchaseFrequency,CustomerLifespan,HistoricalCLTV
0,"12,346.00","6,463.04",6.29,400,"16,259,412.33"
1,"12,347.00",615.19,6.29,402,"1,555,407.99"
2,"12,348.00",403.88,6.29,362,"919,536.64"
3,"12,349.00","1,107.17",6.29,570,"3,969,156.90"
4,"12,350.00",334.40,6.29,0,0.00


## Customer Value Metrics

Customer Lifetime Value (CLTV) metrics were calculated using customer purchase history and transaction data.

The following metrics were derived:

- **Average Order Value (AOV):** Average revenue generated per customer order.
- **Purchase Frequency:** Relative purchasing activity across the customer base.
- **Customer Lifespan:** Number of days between a customer's first and last purchase.
- **Customer Age:** Number of days from the customer's first purchase until the latest transaction in the dataset.
- **Historical CLTV:** Estimated customer lifetime value calculated as:

> Historical CLTV = Average Order Value × Purchase Frequency × Customer Lifespan

These metrics quantify customer value and provide the foundation for customer segmentation and strategic decision-making.

In [20]:
# ==========================================
# Convert Customer Lifespan to Years
# ==========================================

customer_summary["CustomerLifespanYears"] = (
    customer_summary["CustomerLifespan"] / 365
).round(2)

customer_summary[
    [
        "CustomerLifespan",
        "CustomerLifespanYears"
    ]
].head()

,CustomerLifespan,CustomerLifespanYears
0,400,1.10
1,402,1.10
2,362,0.99
3,570,1.56
4,0,0.00


In [21]:
# ==========================================
# Recalculate Historical CLTV
# ==========================================

customer_summary["HistoricalCLTV"] = (
    customer_summary["AverageOrderValue"]
    * customer_summary["PurchaseFrequency"]
    * customer_summary["CustomerLifespanYears"]
).round(2)

customer_summary["HistoricalCLTV"].describe().round(2)

count     5,878.00
mean      1,895.15
std       4,904.48
min           0.00
25%           0.00
50%         883.68
75%       2,604.08
max     296,684.72
Name: HistoricalCLTV, dtype: float64

In [22]:
customer_summary[
    [
        "Customer ID",
        "AverageOrderValue",
        "PurchaseFrequency",
        "CustomerLifespanYears",
        "HistoricalCLTV"
    ]
].head()

,Customer ID,AverageOrderValue,PurchaseFrequency,CustomerLifespanYears,HistoricalCLTV
0,"12,346.00","6,463.04",6.29,1.10,"44,713.38"
1,"12,347.00",615.19,6.29,1.10,"4,256.09"
2,"12,348.00",403.88,6.29,0.99,"2,514.75"
3,"12,349.00","1,107.17",6.29,1.56,"10,862.96"
4,"12,350.00",334.40,6.29,0.00,0.00


## Historical Customer Lifetime Value (CLTV)

Historical Customer Lifetime Value (CLTV) estimates the total value a customer generates throughout their relationship with the business.

For improved interpretability, customer lifespan was converted from days to years before calculating CLTV.

### Formula

> Historical CLTV = Average Order Value × Purchase Frequency × Customer Lifespan (Years)

### Business Significance

- Customers with higher Average Order Value contribute more revenue per purchase.
- Higher Purchase Frequency indicates stronger repeat purchasing behaviour.
- Longer customer lifespans increase long-term customer value.
- The resulting CLTV metric supports customer segmentation and helps identify customers with the greatest long-term business value.

## Historical Customer Lifetime Value (CLTV)

Historical Customer Lifetime Value (CLTV) was estimated using customer purchasing behaviour and relationship duration.

To improve interpretability, customer lifespan was converted from days to years before calculating CLTV.

### Formula

**Historical CLTV = Average Order Value × Purchase Frequency × Customer Lifespan (Years)**

### Key Findings

- The average Historical CLTV is **1,895.15**, while the median is **883.68**, indicating a right-skewed distribution.
- A small group of customers contributes exceptionally high lifetime value, with the highest estimated CLTV reaching **296,684.72**.
- Customers with a lifespan of zero years (one-time purchasers) have a Historical CLTV of zero under this model.
- Historical CLTV provides a quantitative basis for identifying high-value customers and supports customer segmentation for targeted business strategies.

In [23]:
# ==========================================
# Customer Segmentation
# ==========================================

q3 = customer_summary["HistoricalCLTV"].quantile(0.75)

customer_summary["CustomerSegment"] = pd.cut(
    customer_summary["HistoricalCLTV"],
    bins=[-1, 0, q3, customer_summary["HistoricalCLTV"].max()],
    labels=["Low Value", "Medium Value", "High Value"]
)

customer_summary[
    [
        "Customer ID",
        "HistoricalCLTV",
        "CustomerSegment"
    ]
].head()

,Customer ID,HistoricalCLTV,CustomerSegment
0,"12,346.00","44,713.38",High Value
1,"12,347.00","4,256.09",High Value
2,"12,348.00","2,514.75",Medium Value
3,"12,349.00","10,862.96",High Value
4,"12,350.00",0.00,Low Value


> **Note:** Customers with a Historical CLTV of zero are classified as **Low Value**, as they typically represent one-time purchasers with no observed customer lifespan during the analysis period.

In [24]:
# ==========================================
# Segment Distribution
# ==========================================

segment_summary = (
    customer_summary["CustomerSegment"]
    .value_counts()
    .reset_index()
)

segment_summary.columns = ["CustomerSegment", "Customers"]

segment_summary["Percentage"] = (
    segment_summary["Customers"]
    / segment_summary["Customers"].sum()
    * 100
).round(2)

segment_summary

,CustomerSegment,Customers,Percentage
0,Medium Value,2705,46.02
1,Low Value,1703,28.97
2,High Value,1470,25.01


In [25]:
# ==========================================
# Segment Performance
# ==========================================

segment_metrics = (
    customer_summary
    .groupby("CustomerSegment")
    .agg(
        Customers=("Customer ID", "count"),
        AvgRevenue=("TotalRevenue", "mean"),
        AvgOrders=("TotalOrders", "mean"),
        AvgCLTV=("HistoricalCLTV", "mean")
    )
    .round(2)
)

segment_metrics

,Customers,AvgRevenue,AvgOrders,AvgCLTV
CustomerSegment,,,,
Low Value,1703,413.13,1.06,0.00
Medium Value,2705,"1,308.16",5.50,"1,085.58"
High Value,1470,"8,933.80",13.80,"5,580.40"


In [26]:
# ==========================================
# Export Segment Results
# ==========================================

segment_summary.to_csv(
    PROCESSED_DATA / "customer_segments.csv",
    index=False
)

segment_metrics.to_csv(
    PROCESSED_DATA / "segment_metrics.csv"
)

print("Customer segmentation files saved successfully.")

Customer segmentation files saved successfully.


## Customer Segmentation

Customers were segmented based on Historical Customer Lifetime Value (CLTV) to identify groups with different levels of business value.

### Segments

- **Low Value:** Customers with zero or negligible historical lifetime value, typically one-time purchasers.
- **Medium Value:** Customers with moderate historical lifetime value who represent opportunities for increased engagement.
- **High Value:** Customers with the highest historical lifetime value and the greatest long-term revenue contribution.

### Business Value

Customer segmentation enables the business to:

- Prioritize high-value customers for loyalty and retention initiatives.
- Develop targeted campaigns to increase the value of medium-value customers.
- Design reactivation strategies for low-value or one-time customers.
- Allocate marketing resources more efficiently based on customer value.

## Customer Segment Performance

Customer segments were analysed using average revenue, purchasing behaviour and Historical Customer Lifetime Value (CLTV).

### Key Findings

- **High Value customers** generated the highest average revenue ($8,933.80), placed an average of 13.8 orders, and achieved an average Historical CLTV of $5,580.40.
- **Medium Value customers** represented the largest customer group (46.02%) and offer the greatest opportunity for customer growth through targeted marketing and loyalty initiatives.
- **Low Value customers** generated the lowest revenue and typically made only a single purchase, resulting in a Historical CLTV of zero under the adopted methodology.

### Business Recommendations

- Prioritize retention strategies for High Value customers.
- Increase engagement and repeat purchases among Medium Value customers.
- Design cost-effective reactivation campaigns for Low Value customers.

In [27]:
# ==========================================
# Sort Segment Summary
# ==========================================

segment_order = [
    "High Value",
    "Medium Value",
    "Low Value"
]

segment_summary["CustomerSegment"] = pd.Categorical(
    segment_summary["CustomerSegment"],
    categories=segment_order,
    ordered=True
)

segment_summary = (
    segment_summary
    .sort_values("CustomerSegment")
    .reset_index(drop=True)
)

segment_summary

,CustomerSegment,Customers,Percentage
0,High Value,1470,25.01
1,Medium Value,2705,46.02
2,Low Value,1703,28.97


In [28]:
# ==========================================
# Executive KPI Summary
# ==========================================

executive_summary = pd.DataFrame({
    "Metric": [
        "Total Customers",
        "Total Revenue",
        "Total Orders",
        "Average Revenue per Customer",
        "Average Orders per Customer",
        "Average Historical CLTV",
        "High Value Customers",
        "Medium Value Customers",
        "Low Value Customers"
    ],
    "Value": [
        customer_summary.shape[0],
        customer_summary["TotalRevenue"].sum(),
        customer_summary["TotalOrders"].sum(),
        customer_summary["TotalRevenue"].mean(),
        customer_summary["TotalOrders"].mean(),
        customer_summary["HistoricalCLTV"].mean(),
        (customer_summary["CustomerSegment"] == "High Value").sum(),
        (customer_summary["CustomerSegment"] == "Medium Value").sum(),
        (customer_summary["CustomerSegment"] == "Low Value").sum()
    ]
})

executive_summary

,Metric,Value
0,Total Customers,"5,878.00"
1,Total Revenue,"17,374,804.27"
2,Total Orders,"36,969.00"
3,Average Revenue per Customer,"2,955.90"
4,Average Orders per Customer,6.29
5,Average Historical CLTV,"1,895.15"
6,High Value Customers,"1,470.00"
7,Medium Value Customers,"2,705.00"
8,Low Value Customers,"1,703.00"


In [29]:
# ==========================================
# Format Executive Summary
# ==========================================

executive_summary["Value"] = executive_summary["Value"].apply(
    lambda x: f"{x:,.2f}" if isinstance(x, (int, float)) else x
)

executive_summary

,Metric,Value
0,Total Customers,"5,878.00"
1,Total Revenue,"17,374,804.27"
2,Total Orders,"36,969.00"
3,Average Revenue per Customer,"2,955.90"
4,Average Orders per Customer,6.29
5,Average Historical CLTV,"1,895.15"
6,High Value Customers,"1,470.00"
7,Medium Value Customers,"2,705.00"
8,Low Value Customers,"1,703.00"


In [30]:
# ==========================================
# Save Executive Summary
# ==========================================

executive_summary.to_csv(
    PROCESSED_DATA / "executive_summary.csv",
    index=False
)

print("Executive summary exported successfully.")

Executive summary exported successfully.


## Executive KPI Summary

An executive summary was developed to provide a high-level overview of customer behaviour and business performance.

### Key Performance Indicators

- **Total Customers:** 5,878
- **Total Revenue:** 17,374,804.27
- **Total Orders:** 36,969
- **Average Revenue per Customer:** 2,955.90
- **Average Orders per Customer:** 6.29
- **Average Historical CLTV:** 1,895.15

### Customer Segmentation

- High Value Customers: 1,470
- Medium Value Customers: 2,705
- Low Value Customers: 1,703

These KPIs provide a concise business overview and form the basis for executive reporting and dashboard development.

In [31]:
# ==========================================
# Top 10 Customers by Historical CLTV
# ==========================================

top10_cltv = (
    customer_summary
    .sort_values("HistoricalCLTV", ascending=False)
    [
        [
            "Customer ID",
            "HistoricalCLTV",
            "TotalRevenue",
            "TotalOrders",
            "AverageOrderValue",
            "CustomerSegment"
        ]
    ]
    .head(10)
)

top10_cltv

,Customer ID,HistoricalCLTV,TotalRevenue,TotalOrders,AverageOrderValue,CustomerSegment
4061,"16,446.00","296,684.72","168,472.50",2,"84,236.25",High Value
5692,"18,102.00","50,904.71","580,987.04",145,"4,006.81",High Value
68,"12,415.00","44,778.74","144,458.37",28,"5,159.23",High Value
0,"12,346.00","44,713.38","77,556.46",12,"6,463.04",High Value
2277,"14,646.00","44,254.40","528,602.52",151,"3,500.68",High Value
11,"12,357.00","37,189.19","18,287.66",3,"6,095.89",High Value
5109,"17,511.00","36,267.43","172,132.87",60,"2,868.88",High Value
5050,"17,450.00","35,620.78","244,784.25",51,"4,799.69",High Value
4295,"16,684.00","33,484.01","147,142.77",55,"2,675.32",High Value
5841,"18,251.00","31,770.07","26,278.86",9,"2,919.87",High Value


In [32]:
# ==========================================
# Export Top 10 Customers
# ==========================================

top10_cltv.to_csv(
    PROCESSED_DATA / "top10_cltv_customers.csv",
    index=False
)

print("Top 10 CLTV customers exported successfully.")

Top 10 CLTV customers exported successfully.


## Top Customers by Historical CLTV

The top ten customers ranked by Historical Customer Lifetime Value (CLTV) were identified to highlight the organization's most valuable customers.

### Key Findings

- All top ten customers belong to the **High Value** customer segment.
- High customer value can result from either:
  - Frequent purchasing behaviour,
  - High average order values, or
  - A combination of both.
- The highest-value customer achieved a Historical CLTV of **296,684.72**, demonstrating that a small number of customers contribute disproportionately to long-term business value.

### Business Recommendation

Organizations should prioritize personalized engagement, loyalty programmes, and proactive retention strategies for these customers, as losing even a few high-value customers could have a significant impact on long-term revenue.